# Notebook 3 — GRU Model Training & Backtesting

Train the GRU model on the full 32-feature input (OHLCV + technical + advanced indicators + external market data), evaluate on the held-out test set, and run a backtest.

In [10]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.fetcher import DataFetcher, ExternalDataFetcher
from src.data.processor import DataProcessor
from src.models.model_factory import build_model
from src.training.trainer import Trainer
from src.training.evaluator import Evaluator
from src.backtesting.backtest import Backtester
from src.utils.config import load_config

config = load_config()
config['data']['source'] = 'yfinance'
config['data']['timeframe'] = '1Hour'  # 1-hour bars — yFinance supports last ~730 days (2 years)
config['training']['epochs'] = 50   # Lower for notebook speed — raise for production runs
config['model']['type'] = 'gru'     # Switch to 'lstm' to compare

SYMBOL = 'AAPL'
print(f"Model type : {config['model']['type'].upper()}")
print(f"Device     : {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Model type : GRU
Device     : CUDA


In [11]:
# 1. Fetch & process data (OHLCV + 32 engineered features)
fetcher     = DataFetcher(config)
ext_fetcher = ExternalDataFetcher(config)
processor   = DataProcessor(config)

raw_df = fetcher.fetch(SYMBOL, start='2024-02-27', end='2026-02-24')
ext_df = ext_fetcher.fetch(start='2024-02-27', end='2026-02-24')
feat_df = processor.process(raw_df, external_df=ext_df)

config['model']['input_size'] = feat_df.shape[1]
print(f'Feature matrix : {feat_df.shape}')
print(f'Columns        : {feat_df.columns.tolist()}')

[2026-02-25 02:39:56] INFO src.data.fetcher — Loading cached data for AAPL from C:\Users\Arnold\Desktop\Projects\StockPrediction\data\raw\AAPL_1Hour_2024-02-27_2026-02-24.parquet
[2026-02-25 02:39:56] INFO src.data.fetcher — Loading cached external data from C:\Users\Arnold\Desktop\Projects\StockPrediction\data\raw\external_2024-02-27_2026-02-24.parquet
[2026-02-25 02:39:56] INFO src.data.processor — Feature matrix shape after engineering: (3409, 32)
Feature matrix : (3409, 32)
Columns        : ['open', 'high', 'low', 'close', 'volume', 'rsi', 'macd', 'macd_signal', 'macd_diff', 'bb_upper', 'bb_lower', 'bb_width', 'sma_10', 'sma_20', 'sma_50', 'ema_9', 'ema_21', 'volume_change', 'atr', 'obv', 'stoch_k', 'stoch_d', 'williams_r', 'adx', 'vix', 'vix_change', 'yield_10y', 'yield_spread', 'dxy', 'dxy_change', 'spy_rs', 'xlk_rs']


In [12]:
# 2. Build sequences and split
X, y = processor.build_sequences(feat_df, target_col='close')
X_train, y_train, X_val, y_val, X_test, y_test = processor.train_val_test_split(X, y)

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')

[2026-02-25 02:39:56] INFO src.data.processor — Split — train: 2337, val: 500, test: 502
Train: (2337, 70, 32) | Val: (500, 70, 32) | Test: (502, 70, 32)


In [13]:
# 3. Build GRU model and train
model = build_model(config)   # uses config['model']['type'] — 'gru' by default
total_params = sum(p.numel() for p in model.parameters())
print(f'{config["model"]["type"].upper()} | hidden={config["model"]["hidden_size"]} | '
      f'layers={config["model"]["num_layers"]} | params={total_params:,}')

trainer = Trainer(model, config)
history = trainer.train(X_train, y_train, X_val, y_val, symbol=SYMBOL)

GRU | hidden=128 | layers=2 | params=161,409
[2026-02-25 02:39:57] INFO src.training.trainer — Device: cuda | AMP: enabled | cudnn.benchmark: True
[2026-02-25 02:39:57] INFO src.training.trainer — Epoch    1/50 | train_loss=0.025956 | val_loss=0.017837
[2026-02-25 02:39:57] INFO src.training.trainer — Epoch    2/50 | train_loss=0.003203 | val_loss=0.003860
[2026-02-25 02:39:57] INFO src.training.trainer — Epoch    3/50 | train_loss=0.002489 | val_loss=0.003960
[2026-02-25 02:39:58] INFO src.training.trainer — Epoch    4/50 | train_loss=0.002589 | val_loss=0.000725
[2026-02-25 02:39:58] INFO src.training.trainer — Epoch    5/50 | train_loss=0.002141 | val_loss=0.001040
[2026-02-25 02:39:58] INFO src.training.trainer — Epoch    6/50 | train_loss=0.001846 | val_loss=0.000911
[2026-02-25 02:39:58] INFO src.training.trainer — Epoch    7/50 | train_loss=0.001955 | val_loss=0.002138
[2026-02-25 02:39:58] INFO src.training.trainer — Epoch    8/50 | train_loss=0.001797 | val_loss=0.000758
[2026

In [14]:
# Plot training curves
fig = go.Figure()
fig.add_trace(go.Scatter(y=history['train_loss'], name='Train Loss'))
fig.add_trace(go.Scatter(y=history['val_loss'], name='Val Loss'))
fig.update_layout(title='Training & Validation Loss', xaxis_title='Epoch', yaxis_title='MSE Loss')
fig.show()

In [15]:
# 4. Load best checkpoint and evaluate on test set
trainer.load_best(symbol=SYMBOL)
evaluator = Evaluator(model, trainer.device)

print('\n── Test Set Metrics ──')
metrics = evaluator.evaluate(X_test, y_test)

# Get predictions for plotting
y_pred = evaluator.predict(X_test)

[2026-02-25 02:40:07] INFO src.training.trainer — Loaded best checkpoint from C:\Users\Arnold\Desktop\Projects\StockPrediction\models\checkpoints\AAPL_best.pth



── Test Set Metrics ──
[2026-02-25 02:40:07] INFO src.training.evaluator —   mse: 0.000413
[2026-02-25 02:40:07] INFO src.training.evaluator —   rmse: 0.020331
[2026-02-25 02:40:07] INFO src.training.evaluator —   mae: 0.014238
[2026-02-25 02:40:07] INFO src.training.evaluator —   directional_accuracy: 0.522954


In [16]:
# Plot predicted vs actual (normalized scale)
fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test, name='Actual', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=y_pred, name='Predicted', line=dict(color='orange', dash='dash')))
fig.update_layout(title=f'{SYMBOL} — Predicted vs Actual (Test Set, Normalized)',
                  xaxis_title='Time Step', yaxis_title='Normalized Close')
fig.show()

In [17]:
# 5. Inverse transform predictions back to USD
actual_prices = processor.inverse_transform_close(y_test, list(feat_df.columns))
pred_prices = processor.inverse_transform_close(y_pred, list(feat_df.columns))

fig = go.Figure()
fig.add_trace(go.Scatter(y=actual_prices, name='Actual Price', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=pred_prices, name='Predicted Price', line=dict(color='orange', dash='dash')))
fig.update_layout(title=f'{SYMBOL} — Predicted vs Actual Close Price (USD)',
                  xaxis_title='Time Step', yaxis_title='Close Price ($)')
fig.show()

In [18]:
# 6. Backtest
backtester = Backtester(config)
summary, equity_df = backtester.run(actual_prices, pred_prices, threshold=0.002)

print('\n── Backtest Summary ──')
for k, v in summary.items():
    print(f'  {k}: {v}')

[2026-02-25 02:40:07] INFO src.backtesting.backtest — ── Backtest Summary ──────────────────────
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   risk_level: 10.0
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   initial_capital: 10000.0
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   final_equity: 9915.7
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   total_return_pct: -0.84
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   max_drawdown_pct: -1.05
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   sharpe_ratio: -0.782
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   total_trades: 9
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   win_rate_pct: 22.22
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   stop_loss_exits: 4
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   trailing_sl_exits: 4
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   take_profit_exits: 1
[2026-02-25 02:40:07] INFO src.backtesting.backtest —   ha

In [19]:
# Plot equity curve
fig = go.Figure()
fig.add_trace(go.Scatter(x=equity_df['step'], y=equity_df['equity'],
                         name='Portfolio Equity', fill='tozeroy',
                         line=dict(color='green')))
fig.add_hline(y=config['backtesting']['initial_capital'],
              line_dash='dash', line_color='gray', annotation_text='Starting Capital')
fig.update_layout(title=f'{SYMBOL} — Equity Curve (Backtest)',
                  xaxis_title='Time Step', yaxis_title='Portfolio Value ($)')
fig.show()